# Bike Sharing Demand Regression

**Tabular Regression Project** — Predict hourly bike rental demand from weather and temporal features.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Target: `bike_rentals`

## 2 · Project Overview

We predict **hourly bike rental demand** using weather conditions and temporal features. Bike-sharing systems are widespread in cities, and demand forecasting is essential for fleet rebalancing and operational planning.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Build a baseline Linear Regression model.
3. Benchmark many regressors with LazyPredict.
4. Run FLAML AutoML for automated model selection.
5. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
6. Evaluate with RMSE, MAE, R² and residual analysis.
7. Compare all models and select the top 3.

## 4 · Problem Statement

Given the hour of day, weather (temperature, humidity, wind), season, and calendar info (holiday, weekend), predict the **number of bike rentals** for that hour.

## 5 · Why This Project Matters

- **Demand forecasting** enables proactive bike redistribution across stations.
- Weather-aware planning reduces empty-station complaints.
- Understanding commute patterns (morning/evening peaks) optimizes staffing.
- Environmental policy: promoting cycling reduces urban emissions.
- Classic Kaggle-style regression problem with rich feature engineering potential.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,500 |
| **Features** | 10 (hour, temp, humidity, wind, season, weather, holiday, weekend, working day, year) |
| **Target** | `bike_rentals` (count, integer) |
| **Categorical** | season (4), weather_condition (4) |
| **Missing** | None |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Free for learning and experimentation.
- **Limitations**: Simulated data — real-world relationships may be more complex.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "bike_rentals"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

We generate a synthetic dataset so the notebook is fully self-contained.

In [ ]:
np.random.seed(42)
n = 8500
hour = np.random.randint(0, 24, n)
temperature = np.random.uniform(-5, 40, n)
humidity = np.random.uniform(10, 100, n)
wind_speed = np.random.uniform(0, 50, n)
season = np.random.choice(["spring", "summer", "fall", "winter"], n)
weather = np.random.choice(["clear", "cloudy", "light_rain", "heavy_rain"], n, p=[0.4, 0.3, 0.2, 0.1])
is_holiday = np.random.choice([0, 1], n, p=[0.93, 0.07])
is_weekend = np.random.choice([0, 1], n, p=[0.71, 0.29])
working_day = 1 - is_weekend
year = np.random.choice([0, 1], n)

# Hour effect: peak at 8 and 17 (commute hours)
hour_effect = 80 * np.exp(-0.5 * ((hour - 8) / 3) ** 2) + 100 * np.exp(-0.5 * ((hour - 17) / 3) ** 2)

weather_penalty = np.where(weather == "clear", 0, np.where(weather == "cloudy", -30,
                  np.where(weather == "light_rain", -80, -150))).astype(float)
season_effect = np.where(season == "summer", 50, np.where(season == "fall", 30,
                np.where(season == "spring", 10, -40))).astype(float)

rentals = (hour_effect + temperature * 5 - humidity * 1.5 - wind_speed * 2
           + weather_penalty + season_effect + year * 50
           - is_holiday * 30 + working_day * 40
           + np.random.normal(0, 40, n))
rentals = np.maximum(rentals, 0).astype(int)

df = pd.DataFrame({
    "hour": hour, "temperature_c": temperature, "humidity_pct": humidity,
    "wind_speed_kmh": wind_speed, "season": season, "weather_condition": weather,
    "is_holiday": is_holiday, "is_weekend": is_weekend, "working_day": working_day,
    "year": year, "bike_rentals": rentals
})
print(f"Generated dataset: {df.shape}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(["temperature_c", "humidity_pct", "wind_speed_kmh"]):
    axes[i].scatter(df[col], df["bike_rentals"], alpha=0.2, s=8)
    axes[i].set_xlabel(col); axes[i].set_ylabel("bike_rentals")
    axes[i].set_title(f"Rentals vs {col}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby("hour")["bike_rentals"].mean().plot(ax=axes[0], marker="o", color="steelblue")
axes[0].set_title("Mean Rentals by Hour"); axes[0].set_xlabel("Hour"); axes[0].set_ylabel("Rentals")
df.groupby("season")["bike_rentals"].mean().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Mean Rentals by Season"); axes[1].set_ylabel("Rentals")
plt.tight_layout(); plt.show()

## 14 · Target Analysis

Examine the distribution of `bike_rentals`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[TARGET], bins=40, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}"); axes[0].set_xlabel(TARGET)
axes[1].boxplot(df[TARGET], vert=True); axes[1].set_title(f"Box Plot of {TARGET}")
plt.tight_layout(); plt.show()
print(f"Mean: {df[TARGET].mean():,.1f}, Median: {df[TARGET].median():,.1f}, Std: {df[TARGET].std():,.1f}")
print(f"Min: {df[TARGET].min()}, Max: {df[TARGET].max()}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split with a fixed random seed for reproducibility.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(float)

cat_cols = X.select_dtypes(include="object").columns.tolist()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])

X.columns = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing values**: None in this synthetic dataset.
- **Categorical**: Encoded via OrdinalEncoder.
- **Scaling**: Not required for tree-based models.
- **Outliers**: Handled during generation.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()

# Cyclical hour encoding
X_train["hour_sin"] = np.sin(2 * np.pi * X_train["hour"] / 24)
X_train["hour_cos"] = np.cos(2 * np.pi * X_train["hour"] / 24)
X_test["hour_sin"] = np.sin(2 * np.pi * X_test["hour"] / 24)
X_test["hour_cos"] = np.cos(2 * np.pi * X_test["hour"] / 24)

# Comfort index
X_train["comfort_index"] = X_train["temperature_c"] - 0.5 * X_train["humidity_pct"] - 0.3 * X_train["wind_speed_kmh"]
X_test["comfort_index"] = X_test["temperature_c"] - 0.5 * X_test["humidity_pct"] - 0.3 * X_test["wind_speed_kmh"]

print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 18 · Baseline Model

Linear Regression provides a simple, interpretable baseline.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Quickly rank dozens of regressors.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 10 Regressors:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60 s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                     estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                     verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}")
    print(f"R2  : {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML error: {e}")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern gradient-boosting stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# --- CatBoost ---
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    except Exception:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# --- LightGBM ---
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cuda", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cpu", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["LinearRegression"] = y_pred_bl
results["FLAML"] = y_pred_flaml


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)
    axes[i].scatter(y_test, yp, alpha=0.6, s=20, edgecolors="black", linewidths=0.3)
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_pred_vs_actual.png"), dpi=120)
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=25, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

axes[2].scatter(best_pred, np.abs(residuals), alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "error_analysis.png"), dpi=120)
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Hour of day** is the strongest predictor — clear commute peaks at 8 AM and 5 PM.
- **Temperature** has a strong positive effect — warmer weather drives cycling.
- **Weather condition** creates significant penalty — heavy rain cuts demand by ~150 rentals/hour.
- **Humidity** and **wind speed** have moderate negative effects.
- **Season** matters: summer > fall > spring > winter.

**Business takeaway:** Pre-position bikes before morning/evening rush hours. On rainy/cold days, reduce fleet and redirect maintenance resources.

## 26 · Limitations

1. Synthetic data — real bike-sharing has station-level granularity.
2. No spatial dimension — demand varies by neighborhood.
3. Integer target treated as continuous — Poisson regression may be more appropriate.
4. No event data (concerts, sports) that spike demand.
5. Year is binary (0/1) — real trends span multiple years.

## 27 · How to Improve This Project

1. Use real Capital Bikeshare or Citi Bike data.
2. Add station-level features (docks, elevation, transit proximity).
3. Model as count regression (Poisson, Negative Binomial).
4. Add event calendars and public transit disruption data.
5. Use time-series forecasting with lag features.

## 28 · Production Considerations

- Deploy with real-time weather API integration.
- Update predictions hourly for fleet rebalancing.
- Monitor model accuracy during unusual events (storms, holidays).
- Provide probabilistic forecasts for capacity planning.

## 29 · Common Mistakes

1. Treating hour as continuous (0-23) without cyclical encoding.
2. Ignoring the bimodal commute pattern — averaging across hours hides it.
3. Not accounting for weather × hour interactions.
4. Using future weather forecasts without accounting for forecast uncertainty.
5. Overfitting to a single year's patterns.

## 30 · Mini Challenge / Exercises

1. Remove cyclical hour features — does the tree model still capture the commute pattern?
2. Add `hour²` as a feature — compare with sin/cos encoding.
3. Bin weather_condition into good/bad only — how much information is lost?
4. Train separate models for weekdays vs weekends — better or worse?
5. Predict log(1 + bike_rentals) — does it reduce heteroscedasticity?

## 31 · Final Summary / Key Takeaways

1. **Hour of day** dominates demand — commute peaks at 8 AM and 5 PM.
2. **Temperature and weather** are the next most important features.
3. **Cyclical encoding** of hour improves linear models but trees handle it natively.
4. **Gradient boosting** captures the complex hour × weather interactions.
5. **Feature engineering** (comfort index) adds interpretable value.
6. **Count data** may benefit from Poisson-based approaches in production.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))